# Notebook 03: Estrazione Spiegazioni XAI Multi-Granulari
## Fase 3 del Piano Operativo (`results/final_experimental_report.md`)

**Metodologia Formale di Alfredo Nascita et al. (IEEE COMST 2025 & IEEE TNSM 2021):**
1. **T3.1**: SHAP locale con segno preservato ($\pm$) tramite `shap.GradientExplainer` (Expected Gradients / GradientSHAP) sulla matrice spazio-temporale $10 \times 6$ per ciascuno dei 3 modelli su $N=1000$ campioni di test con $N=100$ campioni di background. `shap.DeepExplainer` non rispetta l'additività su queste architetture (ReLU/max-pool funzionali, BatchNorm, softmax dell'attenzione, LayerNorm).
2. **T3.2**: Normalizzazione per istanza Eq. (5) di Nascita et al.:
   $$\tilde{\phi}_{t,f}(x) = \frac{\phi_{t,f}(x)}{\sum_{t'=1}^{10}\sum_{f'=1}^{6} |\phi_{t',f'}(x)|}$$
   preservando tassativamente il segno: $\tilde{\phi} > 0$ (Rosso / Eccitatorio) stimola la classe, $\tilde{\phi} < 0$ (Blu / Inibitorio) allontana dalla classe.
3. **T3.3**: Pooling globale mediano differenziato (campioni correttamente classificati per ciascuna delle 13 classi vs cluster di misclassificazione).
4. **T3.4**: Estrazione dei pesi di attenzione intrinseca (*Explainable-by-Design*): *Attention Rollout* per `BiflowTransformer` e pesi nativi *MHAN* per `MedDefenderTransformer`.
5. **T3.5**: Decomposizione additiva AFA a 3 livelli: Cella ($10 \times 6 = 60$), Campo ($F=6$) e Pacchetto ($T=10$); salvataggio tensori in `results/metrics/xai_explanations.npz`.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import torch
import shap

# Resolve project paths
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
SRC_DIR = REPO_ROOT / 'src'
RESULTS_DIR = REPO_ROOT / 'results'
MODELS_DIR = RESULTS_DIR / 'models'
METRICS_DIR = RESULTS_DIR / 'metrics'
FIGURES_DIR = RESULTS_DIR / 'figures'

for d in [MODELS_DIR, METRICS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils import seed_everything
from util.xai_utils import normalize_shap_by_instance, pool_afa_multiscale
from xai.model_loader import load_checkpoint

seed_everything(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device in uso per XAI: {device}")

### 3.1 Funzioni per l'Estrazione dell'Attenzione Intrinseca By-Design (T3.4)
- `BiflowTransformer`: combina le matrici di attenzione lungo i 2 strati encoder con l'algoritmo di **Attention Rollout**, estraendo la riga del token speciale `[CLS]` verso i 10 pacchetti.
- `MedDefenderTransformer`: estrae i pesi di auto-attenzione a più teste dello strato **MHAN Pooling** sui pacchetti validi, normalizzandoli a somma 1 per ciascun flusso.

In [ ]:
def extract_transformer_rollout(model, x, device='cpu'):
    """
    Calcola Attention Rollout per BiflowTransformer lungo i layer dell'encoder.
    x: tensore [B, 1, T, F] oppure [B, T, F]
    Returns: (pesi [B, T] normalizzati sui T pacchetti, matrice rollout inter-pacchetto [B, T, T]).
    """
    model.eval()
    with torch.no_grad():
        x_dev = x.to(device)
        backbone = model.model if hasattr(model, 'model') else model
        attn_maps = backbone.get_attention_maps(x_dev)[1]

        T = x_dev.size(2) if x_dev.dim() == 4 else x_dev.size(1)
        B = x_dev.size(0)
        rollout = torch.eye(T + 1, device=device).unsqueeze(0).repeat(B, 1, 1)

        for a in attn_maps:
            a_mean = a.mean(dim=1)  # average across attention heads [B, T+1, T+1]
            a_aug = 0.5 * a_mean + 0.5 * torch.eye(a_mean.size(-1), device=device).unsqueeze(0)
            rollout = torch.bmm(a_aug, rollout)  # Abnar & Zuidema: deeper layer goes on the left

        cls_attn = rollout[:, 0, 1:]  # attention from CLS token to the T packets [B, T]
        # Padding mask: zero out attention where all header fields are 0
        x_sq = x_dev.squeeze(1) if x_dev.dim() == 4 else x_dev
        pad_mask = torch.all(x_sq == 0, dim=-1)
        cls_attn = cls_attn.masked_fill(pad_mask, 0.0)
        norm = cls_attn.sum(dim=-1, keepdim=True).clamp(min=1e-12)
        return (cls_attn / norm).cpu().numpy(), rollout[:, 1:, 1:].cpu().numpy()

def extract_meddefender_mhan(model, x, device='cpu'):
    """
    Estrae i pesi di attenzione intrinseca dello strato MHAN pooling per MedDefenderTransformer.
    x: tensore [B, 1, T, F] oppure [B, T, F]
    Returns: (pesi [B, T] per pacchetto, matrice di attenzione inter-pacchetto mediata sulle teste [B, T, T]).
    """
    model.eval()
    with torch.no_grad():
        x_dev = x.to(device)
        backbone = model.model if hasattr(model, 'model') else model
        attn = backbone.get_attention_maps(x_dev)[1]  # [B, nhead, T, T]

        x_sq = x_dev.squeeze(1) if x_dev.dim() == 4 else x_dev
        pad_mask = torch.all(x_sq == 0, dim=-1)  # [B, T]

        # Average across heads and real packet queries (same subset aggregated by masked pooling):
        # padded packets (torch.all(x == 0, dim=-1)) contribute neither as queries nor as keys
        attn_heads = attn.mean(dim=1)  # [B, T, T]
        q_valid = (~pad_mask).unsqueeze(-1).float()  # [B, T, 1]
        attn_mean = (attn_heads * q_valid).sum(dim=1) / q_valid.sum(dim=1).clamp(min=1.0)  # [B, T]
        attn_mean = attn_mean.masked_fill(pad_mask, 0.0)
        norm = attn_mean.sum(dim=-1, keepdim=True).clamp(min=1e-12)
        return (attn_mean / norm).cpu().numpy(), attn_heads.cpu().numpy()

print("[OK] Funzioni di estrazione attenzione intrinseca pronte.")

### 3.2 Caricamento Dataset e Selezione Sottoinsieme Rappresentativo (T3.1)
Estraiamo $N=1000$ flussi stratificati per classe dal test set (che è ordinato per classe) e usiamo i primi $N=100$ come background per SHAP.

In [ ]:
from datasets.data_loader import get_loaders
from xai.model_loader import get_xai_subset

NUM_PKTS = 10
FIELDS = ['PL', 'IAT', 'DIR', 'WIN', 'TTL', 'FLG']
NUM_CLASSES = 13
N_XAI = 1000

_, _, tst_loader, _ = get_loaders(
    datasets=['cic-iomt'],
    num_tasks=1,
    nc_first_task=NUM_CLASSES,
    nc_incr_tasks=0,
    batch_size=128,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    seed=0,
    num_pkts=NUM_PKTS,
    fields=FIELDS
)
test_dataset = tst_loader[0].dataset
print(f"[OK] Test dataset caricato: {len(test_dataset)} campioni.")

# Stratified per-class subset (already shuffled), truncated to N_XAI flows
X_eval, y_eval = get_xai_subset(test_dataset, n_per_class=int(np.ceil(N_XAI / NUM_CLASSES)), seed=0)
X_eval, y_eval = X_eval[:N_XAI], y_eval[:N_XAI]
X_bg = X_eval[:100]
print(f"[OK] Sottoinsieme XAI: X_eval {X_eval.shape}, background X_bg {X_bg.shape}")

### 3.3 Caricamento dei 3 Modelli e Pipeline di Estrazione SHAP + Attenzione (T3.1 - T3.5)
Carichiamo `Lopez17CNN`, `BiflowTransformer` e `MedDefenderTransformer` esclusivamente con `load_checkpoint` (Guida 14.3 regola 2): se un checkpoint manca, il notebook si ferma.

In [ ]:
RUN_IDS = {
    'Lopez17CNN': 'S2_R01_Lopez17CNN_multi_10pkts_seed0',
    'BiflowTransformer': 'S2_R02_BiflowTransformer_multi_10pkts_seed0',
    'MedDefenderTransformer': 'S2_R03_MedDefender_multi_10pkts_seed0'
}

models = {n: load_checkpoint(exp_name=f"cic-iomt_scratch_{e}", results_dir=str(RESULTS_DIR),
                             network=n, device=str(device))[0] for n, e in RUN_IDS.items()}


### 3.4 Calcolo di SHAP (GradientExplainer) con Segno e Normalizzazione Instance-Wise (Eq. 5)
Per ciascun modello, calcoliamo le spiegazioni SHAP con segno per le predizioni sui campioni target e normalizziamo per istanza dividendo per la somma dei valori assoluti.

In [ ]:
import copy
import time

class ModelWrapper(torch.nn.Module):
    """Restituisce i logit della testa (LLL_Net produce una lista di uscite)."""
    def __init__(self, m):
        super().__init__()
        self.m = m
    def forward(self, inp):
        out = self.m(inp)
        return out[0] if isinstance(out, (tuple, list)) else out

def compute_model_shap(model, x_eval, x_bg, n_samples=1000):
    """
    Calcola le attribuzioni SHAP per ciascun campione, preservando il segno algebrico (+/-).
    Returns: (SHAP normalizzato Eq. 5 [N, T, F], classi predette [N]).
    """
    model.eval()
    x_target = x_eval[:n_samples].to(device)
    wrapper = ModelWrapper(model)

    explainer = shap.GradientExplainer(wrapper, x_bg.to(device))
    shap_vals = explainer.shap_values(x_target)
    with torch.no_grad():
        preds = wrapper(x_target).argmax(dim=-1).cpu().numpy()

    if isinstance(shap_vals, list):
        # List of arrays (one per class, shap < 0.45)
        raw_shap = np.stack([shap_vals[preds[i]][i, 0] for i in range(len(x_target))])
    else:
        # Newer format (N, 1, T, F, C): index by each sample's predicted class
        raw_shap = shap_vals[np.arange(len(x_target)), 0, :, :, preds]

    # Instance-wise normalization via Eq. (5) from Nascita et al.
    return normalize_shap_by_instance(raw_shap), preds

def shap_cpu_latency_ms(model, x_bg, x, n_flows=20):
    """Latenza SHAP mediana per singolo flusso (batch 1) su CPU single-thread, come scripts/profile_latency.py."""
    m_cpu = ModelWrapper(copy.deepcopy(model).cpu().eval())
    n_threads = torch.get_num_threads()
    torch.set_num_threads(1)
    try:
        explainer = shap.GradientExplainer(m_cpu, x_bg.cpu())
        times = []
        for i in range(n_flows):
            t0 = time.perf_counter()
            explainer.shap_values(x[i:i + 1].cpu())
            times.append((time.perf_counter() - t0) * 1000)
        return float(np.median(times))
    finally:
        torch.set_num_threads(n_threads)

print("Calcolo SHAP in corso per i 3 modelli...")
N_SHAP_EVAL = N_XAI
shap_explanations = {}
shap_preds = {}
shap_latency_ms = {}
for name, m in models.items():
    print(f"  -> Calcolo SHAP per {name}...")
    shap_explanations[name], shap_preds[name] = compute_model_shap(m, X_eval, X_bg, n_samples=N_SHAP_EVAL)
    shap_latency_ms[name] = shap_cpu_latency_ms(m, X_bg, X_eval)
    print(f"     Forma spiegazioni {name}: {shap_explanations[name].shape} | latenza SHAP CPU 1-thread: {shap_latency_ms[name]:.1f} ms/flusso")

### 3.5 Estrazione Attenzione Intrinseca & Aggregazione a Tre Viste AFA (T3.4 & T3.5)
Estraiamo i vettori temporali di attenzione intrinseca ($T=10$) per `BiflowTransformer` e `MedDefenderTransformer` e decomponiamo SHAP a livello di **Cella**, **Campo** e **Pacchetto**.

In [ ]:
X_subset = X_eval[:N_SHAP_EVAL]

# Zero-cost intrinsic attention extraction
attn_biflow, mat_biflow = extract_transformer_rollout(models['BiflowTransformer'], X_subset, device=device)
attn_meddef, mat_meddef = extract_meddefender_mhan(models['MedDefenderTransformer'], X_subset, device=device)

print(f"[OK] Pesi Attention Rollout Biflow: {attn_biflow.shape}")
print(f"[OK] Pesi MHAN MedDefender:         {attn_meddef.shape}")

# Multi-scale AFA decomposition for each model
afa_multiscale = {}
for name in models.keys():
    cell, field, pkt = pool_afa_multiscale(shap_explanations[name])
    afa_multiscale[name] = {
        'cell': cell,    # [N, 10, 6]
        'field': field,  # [N, 6]
        'pkt': pkt       # [N, 10]
    }
    print(f"[{name}] AFA: Cella {cell.shape} | Campo {field.shape} | Pacchetto {pkt.shape}")

# Differentiated median pooling: correctly classified flows per each of the 13 classes
# vs. misclassified flows cluster (NaN where the group is empty)
y_np = y_eval[:N_SHAP_EVAL].numpy()
def pooled_medians(cell, pred):
    per_class = np.full((NUM_CLASSES, NUM_PKTS, len(FIELDS)), np.nan)
    for c in range(NUM_CLASSES):
        sel = (y_np == c) & (pred == c)
        if sel.any():
            per_class[c] = np.median(cell[sel], axis=0)
    mis = pred != y_np
    mis_med = np.median(cell[mis], axis=0) if mis.any() else np.full((NUM_PKTS, len(FIELDS)), np.nan)
    return per_class, mis_med

pooled = {name: pooled_medians(afa_multiscale[name]['cell'], shap_preds[name]) for name in models}
for name in models:
    print(f"[{name}] T3.3: {int((shap_preds[name] != y_np).sum())} flussi misclassificati su {len(y_np)}")

# Flow-averaged 10x10 matrices averaging only over real (non-padded) packet query rows
valid_q = ~np.all(X_subset.numpy()[:, 0] == 0, axis=-1)  # [N, T]
def masked_mean_matrix(mat):
    w = valid_q[:, :, None].astype(float)
    return (mat * w).sum(axis=0) / np.clip(w.sum(axis=0), 1.0, None)

### 3.6 Salvataggio Tensori XAI in `results/metrics/xai_explanations.npz` (T3.5)
Tutti i tensori estratti vengono persistiti per essere consumati direttamente da Notebook 04, 05 e 06.

In [ ]:
out_npz = METRICS_DIR / 'xai_explanations.npz'
np.savez_compressed(
    out_npz,
    # Cell views (10x6)
    shap_lopez_cell=afa_multiscale['Lopez17CNN']['cell'],
    shap_biflow_cell=afa_multiscale['BiflowTransformer']['cell'],
    shap_meddef_cell=afa_multiscale['MedDefenderTransformer']['cell'],
    # Field views (F=6)
    shap_lopez_field=afa_multiscale['Lopez17CNN']['field'],
    shap_biflow_field=afa_multiscale['BiflowTransformer']['field'],
    shap_meddef_field=afa_multiscale['MedDefenderTransformer']['field'],
    # Temporal packet views (T=10)
    shap_lopez_pkt=afa_multiscale['Lopez17CNN']['pkt'],
    shap_biflow_pkt=afa_multiscale['BiflowTransformer']['pkt'],
    shap_meddef_pkt=afa_multiscale['MedDefenderTransformer']['pkt'],
    # By-design intrinsic attention (T=10)
    attn_biflow_pkt=attn_biflow,
    attn_meddef_pkt=attn_meddef,
    # Flow-averaged 10x10 inter-packet attention matrices
    attn_biflow_mat=masked_mean_matrix(mat_biflow),
    attn_meddef_mat=masked_mean_matrix(mat_meddef),
    # Explained flows: reused in Notebook 04 for Temporal Masking on the same subset
    x_xai=X_subset.numpy(),
    y_xai=y_np,
    # Predicted classes on explained flows (used for per-class RQ1 analysis in Notebook 04)
    pred_lopez=shap_preds['Lopez17CNN'],
    pred_biflow=shap_preds['BiflowTransformer'],
    pred_meddef=shap_preds['MedDefenderTransformer'],
    # Median cell views per correctly predicted class [13, 10, 6] and misclassified cluster [10, 6]
    shap_lopez_cell_class=pooled['Lopez17CNN'][0], shap_lopez_cell_miscls=pooled['Lopez17CNN'][1],
    shap_biflow_cell_class=pooled['BiflowTransformer'][0], shap_biflow_cell_miscls=pooled['BiflowTransformer'][1],
    shap_meddef_cell_class=pooled['MedDefenderTransformer'][0], shap_meddef_cell_miscls=pooled['MedDefenderTransformer'][1],
    # Per-flow SHAP latency on single-thread CPU [ms]
    shap_latency_lopez_ms=shap_latency_ms['Lopez17CNN'],
    shap_latency_biflow_ms=shap_latency_ms['BiflowTransformer'],
    shap_latency_meddef_ms=shap_latency_ms['MedDefenderTransformer'],
    # Metadata
    field_names=np.array(FIELDS),
    num_pkts=NUM_PKTS
)

print(f"\n[SUCCESS] Spiegazioni multi-granulari salvate con successo in:\n  {out_npz}")
print(f"Dimensione archivio compresso: {out_npz.stat().st_size / 1024:.2f} KB")